# 로또 번호 예측 ML 실험

**가설:** 과거 당첨번호 패턴으로 미래 번호를 예측할 수 있는가?  
**결론 선공개:** 없다. 이를 수치로 증명한다.

## 실험 설계
| 모델 | 설명 |
|------|------|
| Baseline (Random) | 1~45에서 무작위 6개 선택 |
| Frequency Model | 출현 빈도 상위 번호 우선 선택 |
| LSTM Model | 직전 N회차 번호를 시퀀스로 학습 |

**평가 지표:** 실제 당첨번호와 겹치는 개수의 평균 (기댓값: 6×6/45 ≈ 0.8)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pandas', 'numpy', 'matplotlib', 'scikit-learn', 'tensorflow'])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA = Path('..') / 'data' / 'lotto.csv'
df = pd.read_csv(DATA)
num_cols = ['n1', 'n2', 'n3', 'n4', 'n5', 'n6']
print(f'데이터: {len(df)}회차')

## 1. 평가 함수 정의

In [ ]:
def hit_count(predicted: set, actual: set) -> int:
    """예측 번호와 실제 번호의 일치 개수"""
    return len(predicted & actual)

def evaluate(predictions: list[set], actuals: list[set]) -> dict:
    hits = [hit_count(p, a) for p, a in zip(predictions, actuals)]
    dist = Counter(hits)
    return {
        'mean_hits': np.mean(hits),
        'std_hits': np.std(hits),
        'distribution': dict(sorted(dist.items())),
        'hits': hits,
    }

# 이론적 기댓값: C(6,k)*C(39,6-k)/C(45,6)
from math import comb
expected = sum(k * comb(6,k) * comb(39,6-k) / comb(45,6) for k in range(7))
print(f'완전 랜덤 이론 기댓값: {expected:.4f}개 일치')

## 2. Train / Test 분할
마지막 100회차를 테스트셋으로 사용

In [ ]:
TEST_SIZE = 100
train_df = df.iloc[:-TEST_SIZE]
test_df  = df.iloc[-TEST_SIZE:]

test_actuals = [set(row) for row in test_df[num_cols].values]
print(f'Train: {len(train_df)}회차 | Test: {len(test_df)}회차')

## 3. Baseline — 완전 랜덤 모델

In [ ]:
rng = np.random.default_rng(42)

random_preds = [set(rng.choice(range(1, 46), size=6, replace=False)) for _ in range(TEST_SIZE)]
random_result = evaluate(random_preds, test_actuals)
print(f'[Random] 평균 일치: {random_result["mean_hits"]:.4f} ± {random_result["std_hits"]:.4f}')

## 4. Frequency Model — 출현 빈도 기반

In [ ]:
all_train_nums = train_df[num_cols].values.flatten()
freq = Counter(all_train_nums)
top6_fixed = set(sorted(freq, key=freq.get, reverse=True)[:6])
print(f'빈도 상위 6번호: {sorted(top6_fixed)}')

freq_preds = [top6_fixed] * TEST_SIZE
freq_result = evaluate(freq_preds, test_actuals)
print(f'[Frequency] 평균 일치: {freq_result["mean_hits"]:.4f} ± {freq_result["std_hits"]:.4f}')

## 5. LSTM Model — 시퀀스 학습

In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)

# 45차원 멀티-핫 인코딩
def to_multihot(row):
    v = np.zeros(45)
    for n in row:
        v[int(n) - 1] = 1
    return v

all_vecs = np.array([to_multihot(row) for row in df[num_cols].values])  # (N, 45)

WINDOW = 10  # 직전 10회차로 다음 회차 예측
X, y = [], []
for i in range(WINDOW, len(all_vecs) - TEST_SIZE):
    X.append(all_vecs[i - WINDOW:i])
    y.append(all_vecs[i])

X = np.array(X)  # (samples, 10, 45)
y = np.array(y)  # (samples, 45)
print(f'학습 샘플: {len(X)}')

In [ ]:
model = keras.Sequential([
    keras.layers.LSTM(64, input_shape=(WINDOW, 45), return_sequences=True),
    keras.layers.LSTM(32),
    keras.layers.Dense(45, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy')
model.summary()

In [ ]:
history = model.fit(
    X, y,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    verbose=1,
)

plt.figure(figsize=(8, 3))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LSTM 학습 곡선')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 테스트셋 예측: 확률 상위 6개를 선택
lstm_preds = []
test_start = len(all_vecs) - TEST_SIZE

for i in range(TEST_SIZE):
    window = all_vecs[test_start + i - WINDOW: test_start + i]
    proba = model.predict(window[np.newaxis], verbose=0)[0]
    top6 = set(np.argsort(proba)[-6:] + 1)  # 1-indexed
    lstm_preds.append(top6)

lstm_result = evaluate(lstm_preds, test_actuals)
print(f'[LSTM] 평균 일치: {lstm_result["mean_hits"]:.4f} ± {lstm_result["std_hits"]:.4f}')

## 6. 결과 비교

In [ ]:
results = {
    'Random (baseline)': random_result,
    'Frequency': freq_result,
    'LSTM': lstm_result,
}

# 평균 일치 수 비교
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 막대 그래프
names = list(results.keys())
means = [r['mean_hits'] for r in results.values()]
stds  = [r['std_hits']  for r in results.values()]
colors = ['#95a5a6', '#3498db', '#e74c3c']

axes[0].bar(names, means, yerr=stds, capsize=5, color=colors)
axes[0].axhline(expected, color='black', linestyle='--', label=f'이론 기댓값 {expected:.3f}')
axes[0].set_ylabel('평균 일치 번호 수')
axes[0].set_title('모델별 평균 적중 수 비교')
axes[0].legend()
axes[0].set_ylim(0, 2)

# 적중 분포 비교
for name, result, color in zip(names, results.values(), colors):
    dist = result['distribution']
    xs = sorted(dist.keys())
    ys = [dist.get(x, 0) / TEST_SIZE * 100 for x in xs]
    axes[1].plot(xs, ys, marker='o', label=name, color=color)

axes[1].set_xlabel('일치 번호 수')
axes[1].set_ylabel('비율 (%)')
axes[1].set_title('일치 번호 수 분포')
axes[1].legend()
plt.tight_layout()
plt.show()

# 수치 요약
print('\n=== 결과 요약 ===')
print(f'이론 기댓값: {expected:.4f}')
for name, r in results.items():
    diff = r['mean_hits'] - expected
    print(f'{name:20s}: {r["mean_hits"]:.4f} (기댓값 대비 {diff:+.4f})')

## 7. 결론

```
세 모델 모두 평균 일치 수가 이론 기댓값(≈0.8)에 수렴한다.
LSTM은 학습을 했음에도 랜덤 베이스라인을 유의미하게 초과하지 못한다.
```

**왜?**
- 로또 추첨은 IID(독립 동일 분포) — 각 회차가 독립 사건
- 과거 데이터에 미래 번호를 예측할 수 있는 패턴이 없음
- 시계열 모델은 자기상관(autocorrelation)이 있는 데이터에만 유효

**포트폴리오 핵심 메시지:**  
*"할 수 없다는 것을 실험으로 증명하는 것도 엄연한 데이터 사이언스다."*